# Exploring the MNIST dataset with PyTorch

Before building a machine-learning model, inspect the data and its context. This notebook explores the size, representation, semantics, visual variation, and class distribution of the MNIST handwritten-digit dataset.

## Learning objectives

After working through this notebook, you should be able to:

- inspect the size, shape, dtype, and value range of a PyTorch vision dataset;
- distinguish the raw stored data from the transformed examples returned by a `Dataset`;
- visualize examples together with their labels;
- compare class distributions across training and test sets;
- identify questions that class counts alone cannot answer.


## Configuration and imports


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torchvision.datasets import MNIST
from torchvision.transforms import v2


In [ ]:
DATA_DIR = Path.cwd() / "data"


## Obtaining the dataset

`torchvision.datasets.MNIST` downloads the dataset when necessary and presents it through PyTorch's `Dataset` interface. Indexing a dataset returns one image-label pair.

The transformation below converts each image to a `float32` tensor and scales pixel values from the integer range 0–255 to the floating-point range 0–1. It preserves the image's channel, height, and width dimensions.


In [ ]:
# TODO: Compose a transform that converts an MNIST image to a
# channels-first float32 tensor whose values are scaled to [0, 1].
transform = ...


In [ ]:
# TODO: Create the training and test MNIST datasets. Reuse `transform`,
# store data below `DATA_DIR`, and allow torchvision to download missing files.
train_dataset = ...
test_dataset = ...


## Dimensions and types

A PyTorch `Dataset` need not store all examples in one array, so its general interface provides a length and indexed access rather than a `.shape` attribute. The MNIST implementation also exposes its raw image and target tensors, which lets us inspect how this particular dataset is stored.


In [ ]:
print(f"Training examples: {len(train_dataset):,}")
print(f"Test examples:     {len(test_dataset):,}")


In [ ]:
print("Raw training images:", train_dataset.data.shape, train_dataset.data.dtype)
print("Training targets:   ", train_dataset.targets.shape, train_dataset.targets.dtype)
print("Raw test images:    ", test_dataset.data.shape, test_dataset.data.dtype)
print("Test targets:       ", test_dataset.targets.shape, test_dataset.targets.dtype)


The raw images are stored as unsigned 8-bit integers with shape `[examples, height, width]`. Targets are 64-bit integer class indices. The transformed representation returned during iteration is different, as the next cell shows.


In [ ]:
image, label = train_dataset[0]

print(f"image type:  {type(image)}")
print(f"image shape: {tuple(image.shape)}")
print(f"image dtype: {image.dtype}")
print(f"value range: [{image.min().item():.1f}, {image.max().item():.1f}]")
print(f"label:       {label}")


## Data semantics

Each input represents a 28 × 28 grayscale image of a handwritten digit. PyTorch uses channels-first image tensors, so the transformed shape is `[1, 28, 28]`: one channel, 28 rows, and 28 columns. The corresponding target is an integer from 0 through 9.


In [ ]:
plt.imshow(image.squeeze(0), cmap="gray", vmin=0.0, vmax=1.0)
plt.title(f"Label: {label}")
plt.axis("off")
plt.show()


A single image cannot show how much handwriting varies. Inspect several examples and compare images assigned the same label. Before running the next cell, predict which digits will show the greatest visual variation.


In [ ]:
ROWS = 5
COLUMNS = 7

figure, axes = plt.subplots(ROWS, COLUMNS, figsize=(7, 5))
for index, axis in enumerate(axes.flat):
    example, target = train_dataset[index]
    axis.imshow(
        example.squeeze(0),
        cmap="gray",
        vmin=0.0,
        vmax=1.0,
    )
    axis.set_title(str(target), fontsize=9)
    axis.axis("off")

figure.suptitle("Training examples; titles show the target labels")
figure.tight_layout()


## Class distribution

All digits should be represented in both splits. Their frequencies matter because aggregate accuracy can be influenced by class imbalance and can conceal poor performance on less frequent or more difficult classes.

`torch.bincount` counts how often each integer class index occurs. Supplying `minlength=10` ensures that an absent class would still appear with count zero.


In [ ]:
# TODO: Count every class in the training and test targets. Ensure that
# classes absent from a split would still appear with a zero count.
train_counts = ...
test_counts = ...

torch.stack([train_counts, test_counts])


In [ ]:
classes = torch.arange(10)
figure, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

for axis, counts, title in zip(
    axes,
    [train_counts, test_counts],
    ["Training set", "Test set"],
):
    axis.bar(classes.numpy(), counts.numpy())
    axis.set(
        title=title,
        xlabel="Digit",
        ylabel="Number of examples",
        xticks=classes.numpy(),
    )

figure.tight_layout()


Because the splits have different sizes, normalized proportions are more appropriate for comparing their class distributions directly.


In [ ]:
train_proportions = train_counts / train_counts.sum()
test_proportions = test_counts / test_counts.sum()

width = 0.4
figure, axis = plt.subplots(figsize=(9, 4))
axis.bar(
    classes.numpy() - width / 2,
    train_proportions.numpy(),
    width,
    label="Training",
)
axis.bar(
    classes.numpy() + width / 2,
    test_proportions.numpy(),
    width,
    label="Test",
)
axis.set(
    xlabel="Digit",
    ylabel="Proportion of examples",
    xticks=classes.numpy(),
)
axis.legend()
figure.tight_layout()


In [ ]:
for name, counts in [
    ("training", train_counts),
    ("test", test_counts),
]:
    smallest_count, smallest_class = counts.min(dim=0)
    largest_count, largest_class = counts.max(dim=0)
    print(
        f"{name:8s}: least frequent={smallest_class.item()} "
        f"({smallest_count.item():,}), most frequent="
        f"{largest_class.item()} ({largest_count.item():,}), "
        f"ratio={largest_count / smallest_count:.3f}"
    )


All ten classes are present, and the training and test proportions are broadly similar, although neither distribution is exactly uniform. These counts alone do not establish that every class is equally easy to recognize or that the two splits contain equally representative handwriting styles. Per-class validation metrics and a confusion matrix remain necessary after training.


## Interpretation questions

1. Why does the transformed image have three dimensions while the raw stored image has two?
2. Which properties of the raw data were changed by the transformation, and which semantic properties were preserved?
3. Are the class frequencies different enough that accuracy becomes meaningless? What additional metric or diagnostic would you report?
4. Does a similar label distribution prove that the training and test images come from the same distribution? What else would you inspect?
5. Why should the test set remain unavailable for model and preprocessing choices even though we explored its aggregate class counts here?
